# 06 — Export the reviewed human-retargeted episode

**Objective:** turn the verified HDF5 recording into a local LeRobot dataset, inspect its inputs, and load a batch of action chunks. No model training or Hub upload occurs.

In [ ]:
!git clone https://github.com/huggingface/lerobot.git
%cd lerobot
!pip install -e ".[libero]"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import shutil
import inspect
import importlib.metadata
import numpy as np
import h5py
import torch
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation
from torch.utils.data import DataLoader
from lerobot.datasets.lerobot_dataset import LeRobotDataset

project = Path("/content/drive/MyDrive/Projects/humanoid-challenge")
SOURCE_RUN = "20261012T161104_719335Z"
source_dir = project / "data/robot/pilot_01" / SOURCE_RUN
report = json.loads((source_dir / "report.json").read_text())
assert report["status"] == "passed_automatic_checks"
assert report["training_eligible"] and not report["visual_review_required"], (
    "Sync the reviewed report.json from the local project before exporting."
)
assert report["visual_review"]["approved"]
assert report["final_task_success"] and report["success_last_second"]
assert report["release_sequence_completed"] and not report["finger_bowl_contact"]
assert report["bowl_plate_contact"]

version = importlib.metadata.version("lerobot")
print("Installed LeRobot:", version)
print("Recorded LeRobot:", report["versions"]["lerobot"])
required = {"repo_id", "fps", "features", "root", "use_videos"}
assert required <= set(inspect.signature(LeRobotDataset.create).parameters)
for method in ["add_frame", "save_episode", "finalize"]:
    assert callable(getattr(LeRobotDataset, method, None)), f"Missing API: {method}"

# Copy once to local Colab disk to avoid thousands of small Drive reads.
work_dir = Path("/content/humanoid_dataset_exports") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
work_dir.mkdir(parents=True, exist_ok=False)
local_h5 = work_dir / "episode.h5"
print("Copying source episode to local Colab disk...")
shutil.copy2(source_dir / "episode.h5", local_h5)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

source_sha256 = sha256_file(local_h5)


In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation

def state9_to_state8(state):
    """XYZ, quaternion XYZW, two finger qpos -> XYZ, rotvec, two finger qpos."""
    state = np.asarray(state, dtype=np.float64)
    if state.shape[-1] != 9 or not np.isfinite(state).all():
        raise ValueError("Expected finite state with last dimension 9")

    quaternions = state[..., 3:7]
    norms = np.linalg.norm(quaternions, axis=-1)
    if not np.allclose(norms, 1, atol=1e-3):
        raise ValueError("Quaternion norms are not close to one")

    normalized = quaternions / norms[..., None]
    rotvec = Rotation.from_quat(normalized.reshape(-1, 4)).as_rotvec()
    rotvec = rotvec.reshape(state.shape[:-1] + (3,))
    return np.concatenate([state[..., :3], rotvec, state[..., 7:9]], axis=-1).astype(np.float32)


def raw_libero_to_policy_features(ob):
    """Use for raw simulator observations at evaluation; images not yet tensorized."""
    raw_state = np.r_[ob["robot0_eef_pos"], ob["robot0_eef_quat"], ob["robot0_gripper_qpos"]]
    return {
        "observation.state": state9_to_state8(raw_state),
        "observation.images.image": np.ascontiguousarray(ob["agentview_image"][::-1]),
        "observation.images.image2": np.ascontiguousarray(ob["robot0_eye_in_hand_image"][::-1]),
    }


## Validate and preview

HDF5 images are already upright; do not flip them again. `done_after` from the source is LIBERO's success signal, not the end of this recorded episode. We retain all frames through release and withdrawal and do not split at first success.


In [ ]:
with h5py.File(local_h5, "r") as f:
    n = int(f.attrs["committed_steps"])
    fps = int(f.attrs["fps"])
    language = f.attrs["language"]

    if isinstance(language, bytes):
        language = language.decode()
    assert n == report["steps"] and n > 0
    assert fps == 20

    for key in ["agentview_image", "wrist_image", "state", "action", "phase", "timestamp_s", "success_after", "done_after"]:
        assert len(f[key]) == n, f"Incomplete rows: {key}"
    assert f["state"].shape == (n, 9)
    assert f["action"].shape == (n, 7)

    for camera in ["agentview_image", "wrist_image"]:
        assert f[camera].shape == (n, 256, 256, 3)
        assert f[camera].dtype == np.uint8

    raw_states = f["state"][:]
    states8 = state9_to_state8(raw_states)
    actions = f["action"][:].astype(np.float32)

    assert np.isfinite(actions).all() and np.max(np.abs(actions)) <= 1.000001
    assert np.allclose(f["timestamp_s"][:], np.arange(n) / fps, atol=1e-6)
    assert np.all(f["success_after"][-fps:])

    phases = f["phase"].asstr()[:]
    assert "release" in phases and "withdraw" in phases

    # Inspect one frame from insertion, transfer and release.
    indices = [int(np.flatnonzero(phases == phase)[0])
               for phase in ["insert", "human_transfer_1", "release"]]
    fig, axes = plt.subplots(3, 2, figsize=(9, 12))
    for row, index in enumerate(indices):
        for col, camera in enumerate(["agentview_image", "wrist_image"]):
            axes[row, col].imshow(f[camera][index])
            axes[row, col].set_title(f"{phases[index]} | frame {index} | {camera}")
            axes[row, col].axis("off")
    plt.tight_layout()
    plt.show()

print("Frames:", n, "| FPS:", fps)
print("Instruction:", language)
print("Converted state:", states8[indices[0]])
print("Paired action:", actions[indices[0]])


## Create the dataset

The `repo_id` below is only a local identifier. No Hub account or upload is needed. We use image features for reliable lossless read-back. LeRobot stores episode boundaries and normalization statistics. State and action arrays remain unnormalized; the training processor will normalize them later.

Action fields remain the executed normalized OSC_POSE commands, not measured displacements in metres. Source phases, raw states, success flags and simulator states remain in the original HDF5 for auditing.


In [ ]:
REPO_ID = "local/humanoid_pilot01_smoke"
dataset_root = work_dir / "lerobot_dataset"
assert not dataset_root.exists(), "Use a new work directory for a fresh export."

features = {
    "observation.images.image": {
        "dtype": "image", "shape": (256, 256, 3),
        "names": ["height", "width", "channels"],
    },
    "observation.images.image2": {
        "dtype": "image", "shape": (256, 256, 3),
        "names": ["height", "width", "channels"],
    },
    "observation.state": {
        "dtype": "float32", "shape": (8,),
        "names": ["eef_x", "eef_y", "eef_z", "rotvec_x", "rotvec_y", "rotvec_z", "finger_left", "finger_right"],
    },
    "action": {
        "dtype": "float32", "shape": (7,),
        "names": ["delta_x", "delta_y", "delta_z", "delta_rot_x", "delta_rot_y", "delta_rot_z", "gripper"],
    },
}

writer = LeRobotDataset.create(
    repo_id=REPO_ID,
    root=dataset_root,
    fps=fps,
    robot_type="libero_panda",
    features=features,
    use_videos=False,
)
conversion_finished = False

try:
    with h5py.File(local_h5, "r") as f:
        for index in range(n):
            writer.add_frame({
                "observation.images.image": f["agentview_image"][index],
                "observation.images.image2": f["wrist_image"][index],
                "observation.state": states8[index].copy(),
                "action": actions[index].copy(),
                "task": language,
            })

            if (index + 1) % 500 == 0:
                print(f"Converted {index + 1}/{n}", flush=True)

    writer.save_episode()
    conversion_finished = True

finally:
    writer.finalize()

assert conversion_finished
print("Dataset saved locally:", dataset_root)


In [ ]:
# Reopen using the same reader training will use.
dataset = LeRobotDataset(repo_id=REPO_ID, root=dataset_root)
assert len(dataset) == n
assert dataset.meta.total_episodes == 1
assert dataset.meta.fps == fps
assert "observation.state" in dataset.meta.stats
assert "action" in dataset.meta.stats

with h5py.File(local_h5, "r") as f:
    for index in [0, n // 2, n - 1]:
        item = dataset[index]
        np.testing.assert_allclose(item["observation.state"].numpy(), states8[index], atol=1e-6)
        np.testing.assert_allclose(item["action"].numpy(), actions[index], atol=1e-7)
        assert item["task"] == language
        for feature, source in [("observation.images.image", "agentview_image"),
                                ("observation.images.image2", "wrist_image")]:
            image = item[feature]
            assert tuple(image.shape) == (3, 256, 256)
            recovered = (image.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)
            np.testing.assert_array_equal(recovered, f[source][index])

# Models such as SmolVLA predict sequences of future actions.
# 50 is the smoke-test chunk length; match it in the training config later.
CHUNK_SIZE = 50
chunked = LeRobotDataset(
    repo_id=REPO_ID, root=dataset_root,
    delta_timestamps={"action": [i / fps for i in range(CHUNK_SIZE)]},
)
batch = next(iter(DataLoader(chunked, batch_size=4, shuffle=False, num_workers=0)))
assert tuple(batch["observation.state"].shape) == (4, 8)
assert tuple(batch["action"].shape) == (4, CHUNK_SIZE, 7)
np.testing.assert_allclose(batch["action"][0].numpy(), actions[:CHUNK_SIZE])
for key in features:
    print(key, tuple(batch[key].shape))
print("Read-back and action-chunk alignment checks passed.")


In [ ]:
# Save the exact preprocessing contract next to the converted data.
preprocessing_source = 'import numpy as np\nfrom scipy.spatial.transform import Rotation\n\n\ndef state9_to_state8(state):\n    """XYZ, quaternion XYZW, two finger qpos -> XYZ, rotvec, two finger qpos."""\n    state = np.asarray(state, dtype=np.float64)\n    if state.shape[-1] != 9 or not np.isfinite(state).all():\n        raise ValueError("Expected finite state with last dimension 9")\n    quaternions = state[..., 3:7]\n    norms = np.linalg.norm(quaternions, axis=-1)\n    if not np.allclose(norms, 1, atol=1e-3):\n        raise ValueError("Quaternion norms are not close to one")\n    normalized = quaternions / norms[..., None]\n    rotvec = Rotation.from_quat(normalized.reshape(-1, 4)).as_rotvec()\n    rotvec = rotvec.reshape(state.shape[:-1] + (3,))\n    return np.concatenate([state[..., :3], rotvec, state[..., 7:9]], axis=-1).astype(np.float32)\n\n\ndef raw_libero_to_policy_features(ob):\n    """Use for raw simulator observations at evaluation; images not yet tensorized."""\n    raw_state = np.r_[ob["robot0_eef_pos"], ob["robot0_eef_quat"], ob["robot0_gripper_qpos"]]\n    return {\n        "observation.state": state9_to_state8(raw_state),\n        "observation.images.image": np.ascontiguousarray(ob["agentview_image"][::-1]),\n        "observation.images.image2": np.ascontiguousarray(ob["robot0_eye_in_hand_image"][::-1]),\n    }\n'
(dataset_root / "pilot_preprocessing.py").write_text(preprocessing_source)
manifest = {
    "source_run": SOURCE_RUN,
    "source_h5_sha256": source_sha256,
    "source_report": report,
    "lerobot_version": version,
    "repo_id": REPO_ID,
    "frames": n,
    "episodes": 1,
    "fps": fps,
    "state_layout": "xyz + scipy principal rotation vector (radians) + two finger qpos",
    "images": "upright RGB, source HDF5 unchanged; image-backed lossless storage",
    "actions": "normalized relative OSC_POSE commands, unchanged",
    "purpose": "training smoke test only; no held-out evaluation split",
    "readback_verified": True,
}
(dataset_root / "conversion_manifest.json").write_text(json.dumps(manifest, indent=2))

# Archive once, then copy to Drive; avoid thousands of small Drive writes.
archive = Path(shutil.make_archive(str(work_dir / "lerobot_dataset"), "zip", root_dir=dataset_root))
export_dir = project / "data/lerobot_exports" / work_dir.name
export_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(archive, export_dir / archive.name)
shutil.copy2(dataset_root / "conversion_manifest.json", export_dir / "conversion_manifest.json")
assert sha256_file(export_dir / archive.name) == sha256_file(archive)

print("Verified Drive archive:", export_dir / archive.name)
print("Local training root:", dataset_root)
print("Keep this local root while the runtime stays connected.")
